<a href="https://colab.research.google.com/github/pmiqueta-oss/redes-neurais-pytorch/blob/main/04_transfer_learning_resnet_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Configurar Dispositivo (GPU/CPU)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando em: {dispositivo}")

# 2. Transformações com Data Augmentation
transformacao = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3), # Converte 1 canal para 3 canais (RGB)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Baixar Dataset MNIST adaptado
dados_treino = datasets.MNIST(root='./data', train=True, download=True, transform=transformacao)
dados_teste = datasets.MNIST(root='./data', train=False, download=True, transform=transformacao)

carregador_treino = DataLoader(dados_treino, batch_size=32, shuffle=True)
carregador_teste = DataLoader(dados_teste, batch_size=32, shuffle=False)

# 4. Carregar modelo PRÉ-TREINADO (ResNet18)
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# CONGELAR parâmetros para aproveitar os pesos da rede pré-treinada
for param in modelo_resnet.parameters():
    param.requires_grad = False

# Substituir apenas a última camada para o nosso problema (10 classes)
num_ftrs = modelo_resnet.fc.in_features
modelo_resnet.fc = nn.Linear(num_ftrs, 10)

modelo_resnet = modelo_resnet.to(dispositivo)

# 5. Perda e Otimizador (Treina APENAS a última camada)
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_resnet.fc.parameters(), lr=0.001)

# 6. Treinamento por Transfer Learning
print("\n--- INICIANDO FINE-TUNING DA RESNET18 ---")
for epoca in range(1, 3):
    modelo_resnet.train()
    perda_total = 0
    for i, (imagens, rotulos) in enumerate(carregador_treino):
        imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)

        otimizador.zero_grad()
        saidas = modelo_resnet(imagens)
        perda = criterio(saidas, rotulos)
        perda.backward()
        otimizador.step()

        perda_total += perda.item()
        if i % 200 == 0:
            print(f"Época {epoca} | Batch {i}/{len(carregador_treino)} - Perda: {perda.item():.4f}")

# 7. Avaliação
modelo_resnet.eval()
corretos, total = 0, 0
with torch.no_grad():
    for imagens, rotulos in carregador_teste:
        imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
        saidas = modelo_resnet(imagens)
        _, previsoes = torch.max(saidas.data, 1)
        total += rotulos.size(0)
        corretos += (previsoes == rotulos).sum().item()

print(f"\nAcurácia do Transfer Learning com ResNet18: {(corretos / total) * 100:.2f}% ✅")

Treinando em: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 345kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.1MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s]



--- INICIANDO FINE-TUNING DA RESNET18 ---
Época 1 | Batch 0/1875 - Perda: 2.4524
Época 1 | Batch 200/1875 - Perda: 0.5469
Época 1 | Batch 400/1875 - Perda: 0.3669
Época 1 | Batch 600/1875 - Perda: 0.2179
Época 1 | Batch 800/1875 - Perda: 0.2470
Época 1 | Batch 1000/1875 - Perda: 0.2771
Época 1 | Batch 1200/1875 - Perda: 0.2942
Época 1 | Batch 1400/1875 - Perda: 0.4969
Época 1 | Batch 1600/1875 - Perda: 0.2896
Época 1 | Batch 1800/1875 - Perda: 0.2067
Época 2 | Batch 0/1875 - Perda: 0.2001
Época 2 | Batch 200/1875 - Perda: 0.1188
Época 2 | Batch 400/1875 - Perda: 0.1284
Época 2 | Batch 600/1875 - Perda: 0.1473
Época 2 | Batch 800/1875 - Perda: 0.1401
Época 2 | Batch 1000/1875 - Perda: 0.1692
Época 2 | Batch 1200/1875 - Perda: 0.2578
Época 2 | Batch 1400/1875 - Perda: 0.3077
Época 2 | Batch 1600/1875 - Perda: 0.0651
Época 2 | Batch 1800/1875 - Perda: 0.1482

Acurácia do Transfer Learning com ResNet18: 95.73% ✅
